In [1]:
import pandas
import matplotlib.pyplot as plt
import numpy
import math
import os
from typing import List
from dataclasses import dataclass


def one_GaussLegendreQuadrature(self, n):
    N = n - 1
    N1 = n
    N2 = n + 1
    xu = numpy.linspace(-1, 1, N1).reshape(-1,1)
    x= - numpy.cos((2*(numpy.linspace(0,N,N1).reshape(-1,1)) + 1)*numpy.pi/(2*N+2)) + (0.27/N1)*numpy.sin(numpy.pi*xu*N/N2)
    L = numpy.zeros((n, n+1))
    x0 = 2

    while numpy.max(numpy.absolute(x-x0)) > numpy.finfo(numpy.float64).eps:
        L[:, 0] = 1
        L[: , 1] = numpy.transpose(x)
        for i in range(1,n):
            temp_Li = ((2*i+1)*(x.T*L[:,i]) - i*L[:,i-1])/(i+1)
            L[:, i+1] = numpy.reshape(temp_Li, (1,-1))
        temp_dLp = ((n)*(x.T*L[:, n]) - (n)*L[:, n-1])/((x.T*x.T) - 1)
        dLp = temp_dLp
        x0 = x
        x = x0 - numpy.reshape((L[:, n]/dLp),(-1,1))
    w = (2/((1-(x.T*x.T))*(dLp*dLp)))
    return x.T, w


class legendreGaussQuad():
    def __init__(self, ngp, dim):
        self.gps1d, self.w1d = self.oneD_GaussLegendreQuadrature(ngp)
        self.gps, self.w, self.ngp = self.multiD_GaussLegendreQuadrature(ngp, dim)
    
    def oneD_GaussLegendreQuadrature(self, n):
        N = n - 1
        N1 = n
        N2 = n + 1
        xu = numpy.linspace(-1, 1, N1).reshape(-1,1)
        x= - numpy.cos((2*(numpy.linspace(0,N,N1).reshape(-1,1)) + 1)*numpy.pi/(2*N+2)) + (0.27/N1)*numpy.sin(numpy.pi*xu*N/N2)
        L = numpy.zeros((n, n+1))
        x0 = 2

        while numpy.max(numpy.absolute(x-x0)) > numpy.finfo(numpy.float64).eps:
            L[:, 0] = 1
            L[: , 1] = numpy.transpose(x)
            for i in range(1,n):
                #using the recurrence relation:
                # (n+1)*P_n+1 = (2n+1)*x*P_n - n*P_n-1
                temp_Li = ((2*i+1)*(x.T*L[:,i]) - i*L[:,i-1])/(i+1)
                L[:, i+1] = numpy.reshape(temp_Li, (1,-1))
            #using the recurrence relation:
            #(1-x^2)*P'_n = (n)*[P_n-1 - x*P_n]
            temp_dLp = ((n)*(x.T*L[:, n]) - (n)*L[:, n-1])/((x.T*x.T) - 1)
            dLp = temp_dLp
            x0 = x
            x = x0 - numpy.reshape((L[:, n]/dLp),(-1,1))
        w = (2/((1-(x.T*x.T))*(dLp*dLp)))
        return x.T, w
    
    def multiD_GaussLegendreQuadrature(self, ngp, dim):
        gps = numpy.zeros((ngp**dim, dim))
        gps[:, 0] = numpy.tile(self.gps1d, ngp**(dim-1))
        if dim == 1:
            w = numpy.reshape(self.w1d, (-1,1))
        elif dim == 3:
            gps[:, 1] = numpy.tile(numpy.repeat(self.gps1d,ngp),ngp)
            gps[:, 2] = numpy.repeat(self.gps1d, ngp**(dim-1))
        else:
            gps[:, 1] = numpy.repeat(self.gps1d, ngp**(dim-1))
            w1 = numpy.tile(self.w1d, ngp**(dim-1))
            w2 = numpy.repeat(self.w1d, ngp**(dim-1))
            w = (w1*w2).T
        
        return gps, w, ngp**dim
    
## quadrature scaling: w_scaled_j = ((b-a)/(d-c))/w_j -> for [c,d] = [-1,1], w_scaled_j = (b-a)/2 * w_j
##                     x_scaled_j = c + (d-c)/(b-a) * (x_j-a) -> for [c,d] = [-1, 1], s_scaled_j = -1 + 2/(b-a) * (x_j-a) 
    def integrate_func(self, func, lim):
        if len(lim) > 1:
            a, b = lim
        else:
            a = 0
            b = lim
        integral = 0
        #print(self.w1d)
        #print(self.gps1d)
        for wx, gpsx in zip(self.w, self.gps):
            #print(gpsx)
            integral += wx*func((b-a)*0.5*gpsx + (b+a)*0.5)
            #print(integral)
        #print('-----', (b-a)*0.5*integral)
        return (b-a)*0.5*integral



In [2]:
quadr = legendreGaussQuad(10,1)

In [ ]:
quadr.gps

In [72]:
#alpha = kappa/(rho*c_p) 
@dataclass
class stefan_variables:
    c_p: float = 1
    kappa: float = 1
    rho: float = 1
    T_l: float = 1
    T_m: float = 1
    h_m: float = 1
    quadrature: legendreGaussQuad = None
    @property
    def alpha(self):
        return self.kappa / (self.rho * self.c_p)

import matplotlib.pyplot as plt
import math
import numpy
from scipy.integrate import quad
from scipy.optimize import fsolve
from scipy import special

def sci_erf(x: float = None):
    return special.erf(x)

def errfunc(x: float = None):
    erf = 2/(math.pi)*numpy.exp(-x**2)
    return erf

#erf(x) := (2/pi) * int^x_0 exp(-1*y**2) dy
def integrate_erf(x: float = None, quadr: legendreGaussQuad = None):
    integral = quadr.integrate_func(errfunc, [0, x])
    return integral 

#lambda is the unique root of the monotonic function,
#f(lambda) = c_p*((T_m - T_l)/h_m) * exp(-1*lambda**2)/(sqrt(pi)*erf(lambda)) - lambda
def return_lambda(vars, lam_init: float = 1.11125):
    stef_num = vars.c_p * ((vars.T_m - vars.T_l)/vars.h_m)
    # = lambda x: c_p*((T_m - T_l)/h_m) * numpy.exp(-1*(x**2))/(numpy.sqrt(math.pi)*integrate_erf(x, quadr)) - x
    def f_lam(lam):
        #erf = integrate_erf(lam, quadr)
        return stef_num * numpy.exp(-1*(lam**2))/(numpy.sqrt(math.pi)*special.erf(lam)) - lam
    roots = fsolve(f_lam, lam_init)
    return roots[0] 

def plot_lam(vars):
    c_p = vars.c_p
    kappa = vars.kappa
    rho = vars.rho
    T_l = vars.T_l
    T_m = vars.T_m
    h_m = vars.h_m
    alpha = vars.alpha
    quadr = vars.quadrature
    def f_l(lam):
            #print(lam)
            erf = integrate_erf(lam, quadr)
            #print(lam, erf)
            return c_p*((T_m - T_l)/h_m) * numpy.exp(-1*(lam**2))/(numpy.sqrt(math.pi)*erf) - lam
    def f_l_np(lam):
            #print(lam)
            erf = integrate_erf(lam, quadr)
            #print(lam, erf)
            return c_p*((T_m - T_l)/h_m) * numpy.exp(-1*(lam**2))/(numpy.sqrt(math.pi)*sci_erf(lam)) - lam
            
    x_val = numpy.linspace(-1.9, 1.9, 100)
    y_val = []
    for x in x_val:
        y_val.append(f_l(x)-f_l_np(x))
    return x_val, y_val
    
#X = 2*sqrt(alpha)*lambda*sqrt(t), for 0 < t < t_end
def interface_position(vars, lam_init: float = 1.7, t_all: List[float] = None, t: float = -7.0):
    if t == -7.0 and t_all is not None:
        X = []
        for idx, t_i in enumerate(t_all):
            lam = return_lambda(vars= vars, lam_init= lam_init)
            X[idx] = 2*numpy.sqrt(vars.alpha*t_i)*lam
        return X
    elif t_all is None and t is not None:
        lam = return_lambda(vars= vars, lam_init= lam_init)
        X = 2*numpy.sqrt(vars.alpha*t)*lam
        return X

#T = T_l - (T_l - T_m)*erf(x/2*sqrt(alpha*t))/erf(X(t)/2*sqrt(alpha*t)) for 0 <= x < X(t), 0 < t < t_end
def temperature_profile(vars, x_all: bool = False, x_loc: float = 100, t: float = -1.0, t_all: List[float] = None):
    def return_single(x_loc, t):
        #print(pos_interface)
        #print(vars.alpha)
        num = special.erf(x_loc/(2*numpy.sqrt(vars.alpha*t)))
        denom = special.erf(pos_interface/(2*numpy.sqrt(vars.alpha*t)))
        return vars.T_l - ((vars.T_l - vars.T_m)*(num/denom))

    if x_all is False:
        if t_all is None:
            pos_interface = interface_position(vars= vars, t= t)
            #print(pos_interface)
            if x_loc < pos_interface:
                return return_single(x_loc, t)
            else:
                return vars.T_m
        
        elif t == -1.0 and t_all is not None:
            T_probe = []
            for t_i in t_all:
                pos_interface = interface_position(vars= vars, t= t_i)
                if x_loc < pos_interface:
                    T_probe.append(return_single(x_loc, t_i))
                else:
                    T_probe.append(vars.T_m)
            return T_probe
    
    elif x_all is True:
        x = numpy.linspace(0, int(x_loc), 1000)
        print('x = linspace(0, ', int(x_loc),', 1000)')
        T_profile = []
        
        if t_all is None:
            pos_interface = interface_position(vars= vars, t= t)
            T_temp = []
            for x_i in x:
                if x_i < pos_interface:
                    T_temp.append(return_single(x_i, t))
                
                else:
                    T_temp.append(vars.T_m)
            T_profile.append(numpy.reshape(T_temp,(1,-1)))

        elif t == -1.0 and t_all is not None:
            for t_i in t_all:
                pos_interface = interface_position(vars= vars, t= t_i)
                T_temp = []
                for x_i in x:
                    if x_i < pos_interface:
                        T_temp.append(return_single(x_i, t_i))
                    
                    else:
                        T_temp.append(vars.T_m)
                T_profile.append(numpy.reshape(T_temp,(1,-1)))

        return T_profile


    

In [21]:
init_vars = stefan_variables(
    c_p = 4200,
    kappa = 0.6,
    rho = 1000,
    T_l = 300,
    T_m = 273,
    h_m = 333700,
    quadrature= quadr
)

val = return_lambda(vars= init_vars)
print(val)

[0.4339115]


C:\Users\harsh\AppData\Local\Temp\ipykernel_17712\75768238.py:42: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  roots = fsolve(f_lam, lam_init)


In [42]:
[print(t, interface_position(vars= init_vars, t= t)) for t in [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 120]]

0 0.0
10 0.0010357456411644232
20 0.0014647655329035445
30 0.0017939640742147842
40 0.0020714912823288465
50 0.002315997661042755
60 0.002537048324164641
70 0.002740325388040208
80 0.002929531065807089
90 0.00310723692349327
100 0.00327531530267103
120 0.0035879281484295683


C:\Users\harsh\AppData\Local\Temp\ipykernel_17712\3871455838.py:42: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  roots = fsolve(f_lam, lam_init)


[None, None, None, None, None, None, None, None, None, None, None, None]

In [73]:
#temperature_profile(vars, x_all: bool = False, x_loc: float = 250.0, t: float = -1.0, t_all: List[float] = None):
T = temperature_profile(vars= init_vars, x_all = True, t_all = numpy.linspace(0, 10000000, 50))

x = linspace(0,  100 , 1000)


C:\Users\harsh\AppData\Local\Temp\ipykernel_17712\727913728.py:42: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  roots = fsolve(f_lam, lam_init)


In [74]:
T

[array([[273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273,
         273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273,
         273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273,
         273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273,
         273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273,
         273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273,
         273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273,
         273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273,
         273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273,
         273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273,
         273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273,
         273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273,
         273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273, 273,
         273, 273, 273, 273, 273, 273,

In [ ]:
x_val = numpy.linspace(-5, 5, 500)
err_val = []
for x in x_val:
    err = integrate_erf(x, init_vars.quadrature) - sci_erf(x)
    print(integrate_erf(x, quadr), sci_erf(x))
    err_val.append(err)

In [7]:
x, y = plot_lam(init_vars)

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(x_val, err_val, label="Function of $\lambda$")
plt.axhline(0, color='red', linestyle='--', label="y=0")
plt.title("Plot of the Function of $\lambda$")
plt.xlabel("$\lambda$")
plt.ylabel("Function Value")
plt.legend()
plt.grid(True)
plt.show()